## Print information about a tuned chorale
1. Chorale name, tolerance, tonal diamond shape, limit max
1. Cent values, note names, scores, and ratios for every chord
2. Top notes cents, note names, cent values


In [2]:
import os
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [3]:
import logging, os, sys, time
from importlib import reload
import numpy as np
from importlib import reload
from collections import Counter, defaultdict
user = 'prent'
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')
base_dir = local_dir
WAVE_DIR = os.path.join(base_dir, 'Music', 'sflib')
# The latest files are here: Archive/straw-man/t1_r1.75_s2.50_md28_sn10/bwv253-opt.npy
numpy_dir = os.path.join(base_dir, 'Archive', 'straw-man')

np.set_printoptions(legacy='1.25')
import diamond_music_utils as dmu
import adaptive_tuning_util as atu 
from itertools import count, combinations, permutations
dmu.start_logger('test.log',log_level = 'info') # how to modify this so that it only prints to the log and not in the notebook.
logging.info(f'{base_dir = }, {numpy_dir = }, {WAVE_DIR = }')
rng = np.random.default_rng()

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [4]:
def print_chords(version, input_file, numpy_dir, measure, tolerance, ratios=True, print_individual_chords=True,\
            offset=0, use_werck_top_notes=False, print_top_notes = True):
    
    _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)
    try:
        floating_cents = np.load(input_file)
        existing_chorale_in_cents = np.rint(floating_cents).astype(int)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    if use_werck_top_notes:
        input_file = os.path.join(numpy_dir, f'{version}-w-top_notes.npy')
    else: 
        input_file = os.path.join(numpy_dir, f'{version}top-notes.npy')
        if print_top_notes:
            print(f'Loaded top_notes from {input_file = }')
    try:      
        top_notes = np.load(input_file)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    top_notes[1] = top_notes[1] + offset
    
    if print_top_notes:
        print(f'Key: {keys[root]} {mode}, {tolerance = }')
        print(f'\ntop notes:')
        print(*[inx for inx in np.arange(12)], sep='\t')
        print(*[note for note in top_notes[0]], sep = '\t')
        print(*[keys[note] for note in top_notes[0]], sep = '\t')
        print(*[cent_value for cent_value in top_notes[1]], sep = '\t')
    if print_individual_chords: 
        print(f'\n#          cents       note names   chord score')
        # #     +----- cents -----+--- note names---+--- chord score'
        # 0:    0  386    0  884	C♮ E♮ C♮ A♮	47.0
    if measure > 0: print(f'\nprinting only measure {measure}')
    prev_chord = np.zeros(4, dtype=int)
    header1 = f" # Fr/To Cents Ratio\t # Fr/To Cents Ratio\t # Fr/To Cents Ratio"
    
    for inx, chord_in_cents, chord_12 in zip(count(0,1), existing_chorale_in_cents.T, chorale.T):
        if not np.array_equal(prev_chord, chord_in_cents):
            if measure == 0 or 16 * (measure - 1) <= inx < 16 * measure:
                if print_individual_chords: 
                        # Join the note names into a single space-separated string to avoid numpy array formatting
                        pitches = ' '.join(map(str, keys[chord_12 % 12]))
                        print(f'{inx}: {atu.format_chord(chord_in_cents,4)}\t{pitches}\t{chord_scorer.score_chord(chord_in_cents, tolerance=tolerance)}')
                if ratios:
                    print(f'{header1}')
                    intervals = []
                    for inx1, inx2 in combinations(np.arange(4),2):
                            cent_value_interval_pair = np.array([chord_in_cents[inx1], chord_in_cents[inx2]])
                            cent_value_delta, cent_value_moves, cent_value_target = atu.cent_value_interval(cent_value_interval_pair)
                            best_idx = chord_scorer.find_best_interval(cent_value_delta, tolerance)[0]
                            ratio = str(atu.limit_format(tonal_diamond[best_idx])[0]).strip()
                            n1 = keys[chord_12[inx1] % 12]
                            n2 = keys[chord_12[inx2] % 12]
                            intervals.append((n1, n2, cent_value_delta, ratio))

                    def fmt(iv, idx):
                            n1, n2, cents, ratio = iv
                            return f"{idx:>2} {n1:>2} {n2:>2} {cents:>5} {ratio:^6}"

                    # print first and last three intervals on separate lines, nicely aligned and without Python punctuation
                    
                    print("   ".join(fmt(iv, i+1) for i, iv in enumerate(intervals[:3])))
                    print("   ".join(fmt(iv, i+1+3) for i, iv in enumerate(intervals[3:])))
        prev_chord = chord_in_cents.copy()
    return keys, root, mode

In [5]:
print(f'{numpy_dir = }')

numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man'


In [19]:
ratio_factors = np.array([ "1.25"]) # , "1.25", "1.75"
stability_factors = np.array(["0"]) # , "1.25"
max_delta = 33
snaps = np.array([0])
# suffixes = np.array(['-opt.npy']) # '-opt.npy',
suffixes = np.array(['-trans-sa-opt.npy'])
suffixes = np.array([
'bwv254_t1_r1.500_lm17-trans-sa-opt.npy',
'bwv259_t2_r1.250_lm19-trans-sa-opt.npy',
'bwv257_t1_r1.375_lm17-trans-sa-opt.npy',
'bwv256_t1_r1.625_lm19-trans-sa-opt.npy',
'bwv255_t2_r1.375_lm19-trans-sa-opt.npy',
'bwv263_t1_r1.250_lm19-trans-sa-opt.npy',
'bwv260_t2_r1.500_lm17-trans-sa-opt.npy',
'bwv258_t1_r1.125_lm19-trans-sa-opt.npy',
'bwv264_t1_r1.625_lm17-trans-sa-opt.npy',
'bwv261_t1_r1.125_lm17-trans-sa-opt.npy',
'bwv253_t1_r1.125_lm17-trans-sa-opt.npy',
'bwv262_t1_r1.750_lm19-trans-sa-opt.npy'])
limit_max = 17
tolerance = 1
measure = 0 # 0 means print all measures
print_individual_chords = True
ratios = True
print_top_notes = False
print_hits_misses = False
use_werck_top_notes = False
total_scores = 0
num_scores = 0
max_score = 0
tonal_diamond = atu.build_tonal_diamond(limit_max)
chord_scorer = atu.ChordScorer(tonal_diamond)

chord_scorer.reset_cache()
# for tolerance in [1]:\n#     for ratio_factor in ratio_factors:
#         for stability_factor in stability_factors:
#             for snap in snaps:
for suffix in suffixes:
    print(f'{suffix = }')
    local_numpy_dir = input_file = os.path.join(numpy_dir, f'viterbi-tunings-5-18')
    version = suffix.split('_')[0]
    tolerance = int(suffix.split('_')[1][1:])
    ratio_factor = float(suffix.split('_')[2][1:])
    # limit_max = int(suffix.split('_')[3][2:].split('-')[0])
    limit_max = int(suffix.split('_')[3][2:].split('-')[0])

    print(f'{local_numpy_dir = }') 
    try:
            #Archive/straw-man/viterbi-tunings-0.90/bwv257_t1_r1.375_lm17-trans-sa-opt.npy
        input_file = os.path.join(local_numpy_dir, f'{suffix}') 
        existing_chorale_in_cents = np.load(input_file)
        logging.info(f'{input_file = }')
    except:
        print(f'Trouble loading {input_file = }')
        continue
    num_scores += 1
    scores = np.array([chord_scorer.score_chord(chord, tolerance=tolerance) for chord in existing_chorale_in_cents.T])
    print(f'\nversion: {version}, Tol: {tolerance}, RF: {ratio_factor}, Average score: {round(np.average(scores),1)}, max score: {np.max(scores)} max chord: {np.argmax(scores)}')
    total_scores += np.average(scores)
    max_score = np.max([max_score, np.max(scores) ])
    keys, root, mode = print_chords(version, input_file, local_numpy_dir, measure, tolerance, \
            ratios=ratios, print_individual_chords=print_individual_chords, \
            use_werck_top_notes=use_werck_top_notes, print_top_notes = print_top_notes)
if print_hits_misses:
    print(f'hits and misses: {chord_scorer.return_cache_results()}')
print(f'overall total: {round(total_scores,1)}, {num_scores = }, Average Score: {round(np.average(total_scores/num_scores),1)}, {max_score = }')

suffix = 'bwv254_t1_r1.500_lm17-trans-sa-opt.npy'
local_numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18'

version: bwv254, Tol: 1, RF: 1.5, Average score: 54.9, max score: 145.0 max chord: 158

#          cents       note names   chord score
0:  192  894  508  192	D♮ A♮ F♮ D♮	45.0
 # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
 1 D♮ A♮   498  4/3      2 D♮ F♮   316  6/5      3 D♮ D♮     0  1/1  
 4 A♮ F♮   386  5/4      5 A♮ D♮   498  4/3      6 F♮ D♮   316  6/5  
4:  508  192  894  192	F♮ D♮ A♮ D♮	45.0
 # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
 1 F♮ D♮   316  6/5      2 F♮ A♮   386  5/4      3 F♮ D♮   316  6/5  
 4 D♮ A♮   498  4/3      5 D♮ D♮     0  1/1      6 A♮ D♮   498  4/3  
6:  508  396  894  192	F♮ E♮ A♮ D♮	82.0
 # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
 1 F♮ E♮   112 16/15     2 F♮ A♮   386  5/4      3 F♮ D♮   316  6/5  
 4 E♮ A♮   498  4/3      5 E♮ D♮   204  9/8      6 A♮

In [128]:
limit_max = 17
tonal_diamond = atu.build_tonal_diamond(limit_max)
chord_scorer = atu.ChordScorer(tonal_diamond)
chord = np.array([0, 316, 583, 933]) # claudes naive proposal 
chord = np.array([596, 315, 933, 0]) # found by sa
print(f'{chord = }')
tolerance = 1
tonal_diamond = atu.build_tonal_diamond(limit_max)
chord_scorer = atu.ChordScorer(tonal_diamond)
score = chord_scorer.score_chord(chord, tolerance=tolerance) 
print(f'{limit_max = }, {tolerance = }, {score = }')

chord = array([596, 315, 933,   0])
limit_max = 17, tolerance = 1, score = 145.0


In [143]:
# from a poll on Facebook in the Microtonal Music and Tuning Theory group.
from fractions import Fraction
tolerance = 8
print(f'{tolerance = }')
# what are the ratios of the main notes in these chords, and how do they score?
# 158:  696  415 1033  100	G♮ E♮ B♭ D♭	145.0
#  # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
#  1 G♮ E♮   281 20/17     2 G♮ B♭   337 17/14     3 G♮ D♭   596 24/17 
#  4 E♮ B♭   582  7/5      5 E♮ D♭   315  6/5      6 B♭ D♭   267  7/6  
# how to get from the ratio format in Chorale-info.ipynb to the ratio format in the poll?
# set the first ratio always to 1:1, then the second to the interval from the first note to the second, 20/17 in this case for G♮ E♮, then the third ratio is set to the second times G♮ B♭   337 17/14, then the fourth to the third times the interval from the G♮ D♭   596 24/17 
#                       +-- fixed at 1:1
#                       |    +-- 1st ratio (1/1) times the interval from 1st to 2nd 20/17
#                       |    |            +-- 2nd ratio times the interval from 1st to 3rd
#                       |    |            |                    +-- 3rd ratio times the interval from 3rd times the ratio from the 3rd note to the 4th note       
ratio_list = np.array([[1/1, 1/1 * 20/17, 1/1 * 20/17 * 17/14, 1/1 * 20/17 * 17/14 * 7/6] ,
                       [10/8, 12/8, 14/8, 17/8], # found by sa score: 145
                       [15/8, 18/8, 21/8, 25/8], # no good
                       [16/8, 19/8, 23/8, 27/8],  # no good
                       [9/8, 11/8, 13/8, 15/8], # best score: 140
                       [35/32,  21/16, 25/16, 15/8],  # no good
                       [1/1, 6/5, 7/5, 12/7]
                       ])
for ratios in ratio_list:
        chord_in_cents = np.array([dmu.ratio_to_cents(ratio).astype(int) for ratio in ratios]) % 1200
        chord_pitch_class = np.array([round(note / 100,0).astype(int) for note in chord_in_cents])
        PC_intervals = [(chord_pitch_class[(num + 1) % 4] - chord_pitch_class[num]) % 12 for num, _ in enumerate(chord_pitch_class)]
        cent_intervals = [(chord_in_cents[(num + 1) % 4] - chord_in_cents[num]) % 1200 for num, _ in enumerate(chord_in_cents)]
        print(f'Ratios: {[str(Fraction(ratio).limit_denominator(max_denominator = 100)) for ratio in ratios]}, cents: {chord_in_cents} notes: {[keys[note] for note in chord_pitch_class]}')
        print(f'Pitch classes: {chord_pitch_class}, PC Intervals: {PC_intervals}, Cent intervals: {cent_intervals}')
        print(f'{chord_in_cents = }')
        score = chord_scorer.score_chord(chord_in_cents, tolerance=tolerance) 
        print(f'{score = }')
        print(f'')

tolerance = 8
Ratios: ['1', '20/17', '10/7', '5/3'], cents: [  0 281 617 884] notes: ['C♮', 'D♯', 'F♯', 'A♮']
Pitch classes: [0 3 6 9], PC Intervals: [3, 3, 3, 3], Cent intervals: [281, 336, 267, 316]
chord_in_cents = array([  0, 281, 617, 884])
score = 145.0

Ratios: ['5/4', '3/2', '7/4', '17/8'], cents: [386 702 968 105] notes: ['E♮', 'G♮', 'A♯', 'C♯']
Pitch classes: [ 4  7 10  1], PC Intervals: [3, 3, 3, 3], Cent intervals: [316, 266, 337, 281]
chord_in_cents = array([386, 702, 968, 105])
score = 145.0

Ratios: ['15/8', '9/4', '21/8', '25/8'], cents: [1088  203  470  772] notes: ['B♮', 'D♮', 'F♮', 'G♯']
Pitch classes: [11  2  5  8], PC Intervals: [3, 3, 3, 3], Cent intervals: [315, 267, 302, 316]
chord_in_cents = array([1088,  203,  470,  772])
score = 2102.0

Ratios: ['2', '19/8', '23/8', '27/8'], cents: [  0 297 628 905] notes: ['C♮', 'D♯', 'F♯', 'A♮']
Pitch classes: [0 3 6 9], PC Intervals: [3, 3, 3, 3], Cent intervals: [297, 331, 277, 295]
chord_in_cents = array([  0, 297, 628, 

In [ ]:
# What about this one, supposed to pass when tolerance is 4:
#     C 	0	—
#     D#	316	6/5 ✓
#     F#	583	7/5 ✓
#     A	    933	12/7 ✓
[1/1, 6/5, 7/5, 12/7], # no good

In [7]:
# Check that cent tunings have not changed the pitch class of any note
print("Checking pitch class preservation...")
violations_found = False
for tolerance in [1]:
    for ratio_factor in ratio_factors:
        for stability_factor in stability_factors:
            for snap in snaps:
                for suffix in suffixes:
                    for version in ['bwv253', 'bwv254', 'bwv255', 'bwv256', 'bwv257', 'bwv258', 'bwv259', 'bwv260', 'bwv261', 'bwv262', 'bwv263', 'bwv264']:
                        try:
                            input_file = os.path.join(local_numpy_dir, f'{version}{suffix}')
                            # input_file = os.path.join(local_numpy_dir, f'{version}-trans-sa-opt.npy') # -trans-sa-opt.npy
                            # print(f'{input_file = }')
                            existing_chorale_in_cents = np.load(input_file)
                        except:
                            print(f'Could not load {input_file}')
                            continue

                        _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)

                        violations = []
                        for chord_inx, (chord_in_cents, chord_12) in enumerate(zip(existing_chorale_in_cents.T, chorale.T)):
                            for voice, (cents, midi) in enumerate(zip(chord_in_cents, chord_12)):
                                original_pc = int(midi) % 12
                                tuned_pc = int(atu.pitch_class_from_cents(cents))  # half-up rounding, consistent with horizontal_transpose.py
                                if original_pc != tuned_pc:
                                    violations.append((chord_inx, voice, int(midi), cents, original_pc, tuned_pc))

                        if violations:
                            violations_found = True
                            print(f'\n{version}: {len(violations)} pitch class violation(s):')
                            for chord_inx, voice, midi, cents, orig_pc, tuned_pc in violations:
                                print(f'  chord {chord_inx}, voice {voice}: MIDI {midi} ({keys[orig_pc]}) -> {cents} cents ({keys[tuned_pc]})')
                        else:
                            print(f'{version}: OK')

if not violations_found:
    print('\nAll pitch classes preserved across all chorales.')

Checking pitch class preservation...
Could not load /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18/bwv253_t1_r1.375_lm17-trans-sa-opt.npy
Could not load /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18/bwv254_t1_r1.375_lm17-trans-sa-opt.npy
Could not load /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18/bwv255_t1_r1.375_lm17-trans-sa-opt.npy
Could not load /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18/bwv256_t1_r1.375_lm17-trans-sa-opt.npy
bwv257: OK
Could not load /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18/bwv258_t1_r1.375_lm17-trans-sa-opt.npy
Could not load /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18/bwv259_t1_r1.375_lm17-trans-sa-opt.npy
Could not load /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/viterbi-tunings-5-18/bwv260_t1_r1.375_lm17-trans-sa-opt.npy
Could n

In [8]:
# Analyze the instrument section feature arrays, created by WreckingCrew.py, which will eventually be used by csound to create the audio output. Each section has its own feature array, which is created by the woodwinds_part, finger_piano_part, melody_part, or the bass_part functions. These four functions are used by different sections of the orchestra. Pick the one you want by setting the feature_array in the next line 


features = {0: 'instrument', 1: 'duration before next note', 2: 'hold_time', 3: 'velocity', 4: 'cents', 5: 'octave', 6: 'voice', 7: 'stereo', 8: 'envelope', 9: 'glissando', 10: 'upsample', 11: 'right_side_envelope', 12: 'second_glissando', 13: 'third_glissando', 14: 'volume'}

all_feature_array_names = ['perc_part_finger_pianos.npy', 'perc_part_pizz_strings.npy', 'perc_part_perc_guitar.npy', 'bass_part_bass_section.npy' , 'melody_part_melody_section.npy', 'winds_part_wood_winds.npy', 'winds_part_brass_section.npy', 'winds_part_bowed_strings.npy']

all_feature_array_names = ['bass_part_bass_section.npy'] # just pick one for now

for feature_array_name in all_feature_array_names:
    feature_array = np.load(feature_array_name, allow_pickle=True)
    # the column locations of the features in the feature_array. 
    
    print(f'section name: {feature_array_name}, note count: {feature_array.shape[0]}')
    for feature in [14, 3]:
        vals, counts = np.unique(np.round(feature_array[:, feature], 2), return_counts=True)
        pairs = [(f'{v:.2f}', int(c)) for v, c in zip(vals, counts)]
        print(f'{features[feature]} values & counts: {pairs}')
        print(f'average: {np.round(np.average(feature_array[:, feature]), 2)}, std: {np.round(np.std(feature_array[:, feature]), 2 )}, min: {np.round(np.min(feature_array[:, feature]), 2)}, max: {np.round(np.max(feature_array[:, feature]), 2)}')
        print(f'') 

FileNotFoundError: [Errno 2] No such file or directory: 'bass_part_bass_section.npy'

In [ ]:
# This cell is specifically targeted at volume and velocity. I want to know how csound will process those features. Looking at this line of csound code: 
#       iamp = ampdb(p4) * p15 / 5 
# that is how csound processes the velocity and volume features. In the features numpy array, those are actually index 14 for volume and 3 for velocity, even though they are 4 and 15 in the csound code. I want to know what each note value for that combination of features using that calculation is actually producing. 

features = {0: 'instrument', 1: 'duration before next note', 2: 'hold_time', 3: 'velocity', 4: 'cents', 5: 'octave', 6: 'voice', 7: 'stereo', 8: 'envelope', 9: 'glissando', 10: 'upsample', 11: 'right_side_envelope', 12: 'second_glissando', 13: 'third_glissando', 14: 'volume'}

all_feature_array_names = ['perc_part_finger_pianos.npy', 'perc_part_pizz_strings.npy',  'bass_part_bass_section.npy' , 'melody_part_melody_section.npy', 'winds_part_wood_winds.npy', 'winds_part_brass_section.npy', 'winds_part_bowed_strings.npy']

# all_feature_array_names = ['perc_part_finger_pianos.npy'] # just pick one for now

for feature_array_name in all_feature_array_names:
    feature_array = np.load(feature_array_name, allow_pickle=True)
    # the column locations of the features in the feature_array. 
    
    print(f'section name: {feature_array_name}, note count: {feature_array.shape[0]}')
    for feature in [1]:
        vals, counts = np.unique(np.round(feature_array[:, feature], 2), return_counts=True)
        pairs = [(f'{v:.2f}', int(c)) for v, c in zip(vals, counts)]
        print(f'{features[feature]} values & counts: {pairs}')
        print(f'average: {np.round(np.average(feature_array[:, feature]), 2)}, std: {np.round(np.std(feature_array[:, feature]), 2 )}, min: {np.round(np.min(feature_array[:, feature]), 2)}, max: {np.round(np.max(feature_array[:, feature]), 2)}')
        print(f'') 

section name: perc_part_finger_pianos.npy, note count: 2031
duration before next note values & counts: [('0.25', 1011), ('0.50', 371), ('0.75', 180), ('1.00', 134), ('1.25', 100), ('1.50', 63), ('1.75', 61), ('2.00', 33), ('2.25', 29), ('2.50', 15), ('2.75', 9), ('3.00', 9), ('3.25', 4), ('3.50', 7), ('3.75', 1), ('4.00', 2), ('4.25', 1), ('4.50', 1)]
average: 0.65, std: 0.61, min: 0.25, max: 4.5

section name: perc_part_pizz_strings.npy, note count: 1984
duration before next note values & counts: [('0.25', 944), ('0.50', 396), ('0.75', 184), ('1.00', 134), ('1.25', 88), ('1.50', 71), ('1.75', 52), ('2.00', 33), ('2.25', 23), ('2.50', 13), ('2.75', 19), ('3.00', 10), ('3.25', 4), ('3.50', 4), ('3.75', 3), ('4.00', 1), ('4.25', 3), ('5.00', 1), ('6.00', 1)]
average: 0.66, std: 0.64, min: 0.25, max: 6.0

section name: bass_part_bass_section.npy, note count: 1767
duration before next note values & counts: [('0.25', 839), ('0.50', 286), ('0.75', 184), ('1.00', 122), ('1.25', 106), ('1.50',

In [ ]:
# Combine all the feature arrays into one. 
all_feature_array_names = [
    'perc_part_finger_pianos.npy', 
    'perc_part_pizz_strings.npy', 
    'bass_part_bass_section.npy',
    'perc_part_marimbas.npy',
    'melody_part_melody_section.npy',
    'winds_part_wood_winds.npy',
    'winds_part_brass_section.npy',
    'winds_part_bowed_strings.npy',
]

arrays = [np.load(name, allow_pickle=True) for name in all_feature_array_names]

# Verify all arrays have 15 columns before combining
for name, arr in zip(all_feature_array_names, arrays):
    assert arr.ndim == 2 and arr.shape[1] == 15, f"{name}: unexpected shape {arr.shape}"

combined = np.vstack(arrays)
np.save('all_sections_combined.npy', combined)

print(f'Combined shape: {combined.shape}')
for name, arr in zip(all_feature_array_names, arrays):
    print(f'  {name}: {arr.shape[0]} notes')
print(f'  Total: {combined.shape[0]} notes, {combined.shape[1]} features')


Combined shape: (9604, 15)
  perc_part_finger_pianos.npy: 2031 notes
  perc_part_pizz_strings.npy: 1984 notes
  bass_part_bass_section.npy: 1767 notes
  perc_part_marimbas.npy: 1927 notes
  melody_part_melody_section.npy: 483 notes
  winds_part_wood_winds.npy: 385 notes
  winds_part_brass_section.npy: 537 notes
  winds_part_bowed_strings.npy: 490 notes
  Total: 9604 notes, 15 features


In [ ]:
# Now compute the start_time feature, which is not included in the original feature arrays. The start_time is the cumulative sum of the durations before the next note (column 1) for each voice (column 6). We will add this as a new column at the end of the combined array.
all_feature_array_names = [
    'perc_part_finger_pianos.npy', 
    'perc_part_pizz_strings.npy', 
    'bass_part_bass_section.npy',
    'perc_part_marimbas.npy',
    'melody_part_melody_section.npy',
    'winds_part_wood_winds.npy',
    'winds_part_brass_section.npy',
    'winds_part_bowed_strings.npy',
]

arrays = [np.load(name, allow_pickle=True) for name in all_feature_array_names]
combined = np.vstack(arrays)

# Compute start_time per voice (column 6), using duration before next note (column 1)
start_times = np.zeros(combined.shape[0])
for voice in np.unique(combined[:, 6]):
    mask = combined[:, 6] == voice
    durations = combined[mask, 1]
    start_times[mask] = np.concatenate([[0], np.cumsum(durations[:-1])])

combined = np.column_stack([combined, start_times])
np.save('all_sections_combined.npy', combined)

features = {0: 'instrument', 1: 'duration before next note', 2: 'hold_time', 3: 'velocity',
            4: 'cents', 5: 'octave', 6: 'voice', 7: 'stereo', 8: 'envelope', 9: 'glissando',
            10: 'upsample', 11: 'right_side_envelope', 12: 'second_glissando',
            13: 'third_glissando', 14: 'volume', 15: 'start_time'}

print(f'Combined shape: {combined.shape}')
# Spot-check one voice
v = int(combined[0, 6])
mask = combined[:, 6] == v
print(f'\nVoice {v} first 5 notes:')
print(f'  durations:   {combined[mask, 1][:5]}')
print(f'  start_times: {combined[mask, 15][:5]}')


Combined shape: (9604, 16)

Voice 0 first 5 notes:
  durations:   [1.5  1.25 0.25 1.75 0.25]
  start_times: [0.   1.5  2.75 3.   4.75]


In [ ]:
# Filter to only rows that produce sound: ampdb(velocity) * volume / 5 > 0
velocity = combined[:, 3]
volume   = combined[:, 14]

amp = (10 ** (velocity / 20)) * volume / 5
playing_mask = amp >= 1.0

playing_notes = combined[playing_mask]
np.save('playing_notes.npy', playing_notes)

print(f'Original rows:  {combined.shape[0]}')
print(f'Playing rows:   {playing_notes.shape[0]}')
print(f'Discarded rows: {combined.shape[0] - playing_notes.shape[0]}')

# Sanity check
remaining_amp = (10 ** (playing_notes[:, 3] / 20)) * playing_notes[:, 14] / 5
print(f'\nAmplitude range: min={remaining_amp.min():.2f}, max={remaining_amp.max():.2f}')


Original rows:  9604
Playing rows:   8465
Discarded rows: 1139

Amplitude range: min=8.41, max=12497.56


In [ ]:
# inspect the low amplitude notes to look for outliers, especially notes that are too quiet to be heard, but may be perceived as noise by the listener. I'd like to understand what the threshold should be.
low_amp_notes = 0
min_amp_threshold = 10
notes_to_check = min(20000, playing_notes.shape[0])
for i in np.arange(notes_to_check): 
    velocity = playing_notes[i, 3]
    volume = playing_notes[i, 14]
    amp = (10 ** (velocity / 20)) * volume / 5

    if amp < min_amp_threshold:
        print(f'Note {i} has velocity {playing_notes[i, 3].astype(int)} and volume {playing_notes[i, 14].astype(int)}, resulting in amplitude {amp:.2f} which is below the threshold of {min_amp_threshold}.')
        low_amp_notes += 1  
print(f'{low_amp_notes} notes out of {notes_to_check} have amplitude below {min_amp_threshold}')

Note 4203 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4205 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4206 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4210 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4211 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4213 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4214 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4215 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4216 has velocity 70 and volume 0, resulting in amplitude 8.41 which is below the threshold of 10.
Note 4217 has velocity 70 and volume 0, resulting in amplitude 8